# 06 · Expression - EMT-TF readout of the arms

The divergent arms out of the hub are demarcated by distinct EMT transcription
factors. Selection is **data-driven**: from a candidate EMT-TF panel, pick the
two most arm-specific factors and overlay cells above a data-driven expression
threshold on the manifold. Deliverable: `expression.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
# cwd = <root>/mesencontext/claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*4))                   # -> EMToolbox root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'mesencontext'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'mesencontext', 'claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

In [3]:
adata = sc.read_h5ad(os.path.join(pth_out, 'processed.h5ad'))
cand = [x for x in ['ZEB1','ZEB2','PRRX1','PRRX2','SNAI1','SNAI2','TWIST1','TWIST2']
        if x in adata.var_names]
Xc = pd.DataFrame({x: np.asarray(adata[:,x].X.todense()).ravel() for x in cand},
                  index=adata.obs_names)
Xc['ct'] = adata.obs.celltype.values
prof = Xc.groupby('ct', observed=True).mean()
print('EMT-TF mean expression by subtype:'); print(prof.round(2).T.to_string())
# Data-driven pick: the most ARM-SPECIFIC factors, not merely high-variance.
# specificity = top-arm mean / total arm expression (concentration in one arm);
# require the top arm to be meaningfully expressed, and take the single best TF
# for each distinct arm so the chosen pair demarcates DIFFERENT branches.
arms = prof.drop(index='Mixed', errors='ignore')
spec = (arms.max() / (arms.sum() + 1e-9)); top_arm = arms.idxmax()
expressed = arms.max() >= max(0.5, float(arms.values.mean()))
cand_rank = spec[expressed].sort_values(ascending=False)
emt, seen_arm = [], set()
for tf in cand_rank.index:
    a = top_arm[tf]
    if a in seen_arm: continue
    emt.append(tf); seen_arm.add(a)
    if len(emt) == 2: break
palette = ['tab:red','tab:blue']
g_plt = {tf: (palette[i], float(np.quantile(Xc[tf], .90)))
         for i, tf in enumerate(emt)}
print('arm specificity:', spec.round(2).to_dict())
print('EMT-TFs shown (arm):', {tf: top_arm[tf] for tf in emt})
print('EMT-TFs shown:', g_plt)

EMT-TF mean expression by subtype:
ct      Atypical  Mixed  Basal  Classical
ZEB1        0.03   0.23   0.12       1.74
ZEB2        0.08   0.05   0.10       0.05
PRRX1       1.40   0.20   0.06       0.66
PRRX2       0.00   0.07   0.06       0.12
SNAI1       0.65   0.21   0.46       0.18
SNAI2       1.65   1.86   4.30       3.44
TWIST1      0.32   0.30   0.21       0.03
TWIST2      0.20   0.06   0.17       0.13
arm specificity: {'ZEB1': 0.92, 'ZEB2': 0.44, 'PRRX1': 0.66, 'PRRX2': 0.66, 'SNAI1': 0.5, 'SNAI2': 0.46, 'TWIST1': 0.58, 'TWIST2': 0.4}
EMT-TFs shown (arm): {'ZEB1': 'Classical', 'PRRX1': 'Atypical'}
EMT-TFs shown: {'ZEB1': ('tab:red', 3.334936977139168), 'PRRX1': ('tab:blue', 1.8091288307132145)}


### EMT-TF expression overlay (deliverable)

In [4]:
fig, ax = plt.subplots(1, 1, figsize=(5,5))
sc.pl.umap(adata, size=180, alpha=.5, frameon=False, show=False, ax=ax)
for var, (c, thresh) in g_plt.items():
    sub = adata[np.asarray(adata[:,var].X.todense()).ravel() > thresh].copy()
    sc.pl.umap(sub, na_color=c, size=180, alpha=.5, frameon=False, show=False, ax=ax)
sc.pl.umap(adata, color='celltype', size=180, alpha=0, legend_loc='on data',
           legend_fontoutline=1, frameon=False, show=False, ax=ax)
ax.invert_yaxis(); ax.invert_xaxis()
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([],[],marker='o',ls='',color=c,label=v)
          for v,(c,_) in g_plt.items()], frameon=False, loc='lower right', fontsize=9)
ax.set_title('EMT-TF Expression', size=13)
fig.savefig(os.path.join(pth_fig,'expression.pdf'), bbox_inches='tight')